# 1. RunnableLambda

`RunnableLambda` turns **any Python function** into a Runnable so it can live inside a chain. It's how
you inject your *own* logic — reshaping data, cleaning text, doing math, calling an API — between the
big prompt/model/parser steps.

---

## 1. Simple Definition

> **Kid version:** Your chain is a train 🚂 where each carriage does a job. Most carriages are pre-built
> (prompt, model, parser). But sometimes you want a carriage that does **your own special trick** — like
> flipping text to uppercase. `RunnableLambda` is a **build-your-own carriage kit**: put your function
> inside, and now it couples onto the train like any other carriage.

**Professional definition:** `RunnableLambda` wraps a plain Python callable in the `Runnable`
interface, giving it `.invoke()`, `.stream()`, `.batch()`, and async support, so it can be composed with
other Runnables using the `|` operator.

```python
from langchain_core.runnables import RunnableLambda

to_upper = RunnableLambda(lambda x: x.upper())
to_upper.invoke("hello")        # "HELLO"
```

---

## 2. Why Does It Exist?

**The problem:** A chain only knows how to connect **Runnables**. But your custom logic is just a normal
function — it doesn't speak `invoke`/`stream`/`batch`. Without a wrapper, you'd have to *break out* of the
chain to run it, losing composability, streaming, and batching.

### Before (break the chain to run custom logic)

```python
summary = summarize_chain.invoke({"text": article})   # leave the chain...
payload = {"summary": summary}                          # ...do plain Python by hand...
result  = translate_chain.invoke(payload)               # ...re-enter. Not one chain anymore.
```

### After (wrap the logic as a step)

```python
chain = summarize_chain | RunnableLambda(lambda s: {"summary": s}) | translate_chain
chain.invoke({"text": article})   # custom reshape happens INSIDE the chain
```

`RunnableLambda` lets arbitrary Python live **inside** the pipeline, so the whole thing stays a single
composable, streamable, batchable Runnable.

**Where you'll use it:** reshaping one step's output into the next step's input, text pre/post-processing,
small calculations, calling external APIs/DB lookups, adding logging, and **dynamic routing** (returning
a chain to run).

---

## 3. Real-Life Analogy

**A custom adapter on an assembly line** 🔌. The big machines are fixed, but between them you clip in a
small custom jig that trims a part, relabels it, or rotates it so the next machine accepts it. Tiny, but
the line won't flow without it. `RunnableLambda` is that made-to-order adapter.

---

## 4. Where It Fits in LangChain Architecture

```
                       Runnable  (shared interface)
                            │
   ┌──────────┬─────────────┼───────────────┬───────────────┐
   ▼          ▼             ▼               ▼               ▼
 Prompt      Model        Parser      RunnableLambda    RunnablePassthrough
                                     (your function)     
```

`RunnableLambda` sits right beside prompts/models/parsers as a first-class chain step — the difference is
*you* supply the behavior. It's the general-purpose primitive; `RunnablePassthrough` is
essentially a specialized cousin.

---

## 5. Internal Working

```
  chain = step_a | RunnableLambda(f) | step_b
  chain.invoke(x)

  ① step_a produces output_a
        │
        ▼
  ② RunnableLambda calls f(output_a)          ← your function runs here
        │  returns transformed value
        ▼
  ③ that value becomes step_b's input
```

Key facts about the wrapped function:
- It takes **exactly one argument** — the input from the previous step (use a dict to pass "multiple"
  values).
- Its **return value** becomes the next step's input (so its output type must fit downstream).
- If it returns a **Runnable**, LangChain will *invoke that Runnable* on the input — the basis of dynamic
  routing.

---

## 6. Ways to create one

### Explicit wrapping

**Definition:** Pass any callable to `RunnableLambda(...)`.

**Why it exists:** Clear, and lets you call `.invoke()`/`.batch()` on it standalone.

**When developers use it:** When they want an obvious, reusable step.

In [1]:
from langchain_core.runnables import RunnableLambda

def clean(text: str) -> str:
    return text.strip().lower()

clean_step = RunnableLambda(clean)
clean_step.invoke("  HELLO  ")

'hello'

In [2]:
from langchain_core.runnables import RunnableLambda

def count_words(text: str) -> int:
    return len(text.split())

count_step = RunnableLambda(count_words)

count_step.invoke("LangChain makes building AI applications easier")

6

### Implicit (auto-wrapping inside a chain)

**Definition:** A bare function/lambda placed in a `|` chain is **automatically** coerced to a
`RunnableLambda`.

**Why it exists:** Convenience — less boilerplate for quick transforms.

**When developers use it:** Small inline reshapes.

```python
# the lambda is auto-wrapped:
chain = summarize_chain | (lambda s: {"summary": s}) | translate_chain
```

> Use the **explicit** form when you want to reuse the step, unit-test it, or call it on its own; use the
> **implicit** form for quick inline glue.

In [3]:
from langchain_core.runnables import RunnableLambda

def get_topic(text: str) -> str:
    return text.strip().lower()

topic_chain = RunnableLambda(get_topic)

def format_topic(data: dict) -> str:
    return f"The topic is: {data['topic']}"

format_chain = RunnableLambda(format_topic)

chain = topic_chain | (lambda s: {"topic": s}) | format_chain

result = chain.invoke("  Artificial Intelligence  ")

print(result)

The topic is: artificial intelligence


In [4]:
from langchain_core.runnables import RunnableLambda

def extract_name(text: str) -> str:
    return text.strip().title()

name_chain = RunnableLambda(extract_name)

def create_greeting(data: dict) -> str:
    return f"Hello, {data['name']}! Welcome to LangChain."

greeting_chain = RunnableLambda(create_greeting)

chain = name_chain | (lambda name: {"name": name}) | greeting_chain

result = chain.invoke("  rahul  ")

print(result)

Hello, Rahul! Welcome to LangChain.


### The @chain decorator (bonus)

**Definition:** `@chain` turns a function into a Runnable, letting you write multi-step custom logic as
one function.

**Why it exists:** Sometimes a custom function *is* the chain (with its own internal `.invoke()` calls).

```python
from langchain_core.runnables import chain

@chain
def my_step(inputs: dict) -> dict:
    text = inputs["text"].strip()
    return {"clean": text, "length": len(text)}

my_step.invoke({"text": "  hi "})   # {"clean": "hi", "length": 2}
```

In [5]:
from langchain_core.runnables import chain

@chain
def analyze_text(inputs: dict) -> dict:
    text = inputs["text"].strip()
    words = text.split()
    return {"text": text, "word_count": len(words), "uppercase": text.upper()}

result = analyze_text.invoke({"text": "  LangChain is powerful  "})

print(result)

{'text': 'LangChain is powerful', 'word_count': 3, 'uppercase': 'LANGCHAIN IS POWERFUL'}


In [6]:
from langchain_core.runnables import chain

@chain
def calculate_discount(inputs: dict) -> dict:
    price = inputs["price"]
    discount = inputs["discount"]
    final_price = price - (price * discount / 100)
    return {"original_price": price, "discount": discount, "final_price": final_price}

result = calculate_discount.invoke({"price": 1000, "discount": 20})

print(result)

{'original_price': 1000, 'discount': 20, 'final_price': 800.0}


## 7. Common patterns

### Reshape output → next input (the #1 use)

```python
# stage 1 emits a string; stage 2's prompt wants {"summary": ...}
chain = summarize_chain | RunnableLambda(lambda s: {"summary": s}) | translate_chain
```

---

### Pre/post-processing

```python
# Post-process a model's text answer
chain = prompt | model | StrOutputParser() | RunnableLambda(str.strip)
```

---

### Call an external system mid-chain

```python
def enrich(inputs: dict) -> dict:
    inputs["stock"] = get_price(inputs["ticker"])   # your API/DB call
    return inputs

chain = RunnableLambda(enrich) | prompt | model
```

---

### Dynamic routing (return a Runnable to run)

**Definition:** A `RunnableLambda` can inspect the input and **return the chain to use**; LangChain then
invokes that chain on the input.

**Why it exists:** Flexible if/else routing driven by arbitrary Python.

```python
from langchain_core.runnables import RunnableLambda

def route(info: dict):
    if info["topic"] == "math":
        return math_chain          # returns a Runnable
    return general_chain

full = RunnableLambda(route)
full.invoke({"topic": "math", "q": "2+2"})   # runs math_chain on the input
```

(For explicit condition/branch routing, see `RunnableBranch` in the Chains notebook.)

In [8]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def create_result(text):
    return {"joke": text, "word_count": len(text.split())}

prompt = PromptTemplate(template="Write a joke about {topic}",
                        input_variables=["topic"])

llm = ChatOllama(model="qwen3:8b")

parser = StrOutputParser()

joke_gen_chain = prompt | llm | parser

final_chain = joke_gen_chain | RunnableLambda(create_result)

result = final_chain.invoke({"topic": "AI"})

final_result = "{} \n word count - {}".format(result["joke"], result["word_count"])

print(final_result)

Why did the AI get kicked out of the café?  
It tried to "optimize" the coffee order by brewing a latte instead of a regular coffee. The barista said, "You’re too literal—this isn’t a math problem!"  

*(Bonus: The AI just learned to add "espresso" to its vocabulary.)* ☕🤖 
 word count - 48
